# INTCALC parity check: COBOL `CBACT04C` vs Java `intcalc`

Compares the outputs of the original GnuCOBOL run of `CBACT04C` and the Java port on the same input data:

1. **Row counts** of the interest transaction (SYSTRAN) file and the rewritten account file
2. **Interest per account** (sum of `TRAN-AMT`) and resulting `ACCT-CURR-BAL`
3. **Row-level diff** of every field of every record
4. **PASS / FAIL** verdict

Inputs are the raw fixed-length outputs (`RECFM=F`): 350-byte `TRAN-RECORD`s (`CVTRA05Y`) and 300-byte `ACCOUNT-RECORD`s (`CVACT01Y`). They are decoded here from the copybook layouts, independently of the Java code.

Runs on Databricks (set the `base_path` widget to a DBFS / Unity Catalog volume path containing `<dataset>/cobol_*.dat` and `<dataset>/java/*.dat`) or locally with PySpark.

In [1]:
import os
from decimal import Decimal

IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IS_DATABRICKS:
    dbutils.widgets.text("base_path", "/Volumes/carddemo/intcalc/parity")
    dbutils.widgets.text("datasets", "sample,carddemo-seed")
    BASE_PATH = dbutils.widgets.get("base_path")
    DATASETS = [d.strip() for d in dbutils.widgets.get("datasets").split(",") if d.strip()]
else:
    from pyspark.sql import SparkSession
    spark = (SparkSession.builder.master("local[2]").appName("intcalc-parity")
             .config("spark.ui.enabled", "false").config("spark.ui.showConsoleProgress", "false").getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")
    BASE_PATH = os.path.abspath(os.environ.get("INTCALC_PARITY_OUTPUT", "../output"))
    DATASETS = os.environ.get("INTCALC_PARITY_DATASETS", "sample,carddemo-seed").split(",")

RUN_MODE = (f"DATABRICKS (runtime {os.environ.get('DATABRICKS_RUNTIME_VERSION')})" if IS_DATABRICKS
            else f"LOCAL RUN - PySpark {spark.version} on this machine, NOT Databricks")
print("=" * 78)
print(f"RUN MODE : {RUN_MODE}")
print(f"BASE PATH: {BASE_PATH}")
print(f"DATASETS : {DATASETS}")
print("=" * 78)

26/10/30 20:51:19 WARN Utils: Your hostname, devin-box resolves to a loopback address: 127.0.0.1; using 172.16.255.2 instead (on interface ens5)
26/10/30 20:51:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/30 20:51:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


RUN MODE : LOCAL RUN - PySpark 3.5.1 on this machine, NOT Databricks
BASE PATH: /home/ubuntu/repos/aws-mainframe-modernization-carddemo/modernization/intcalc/parity/output
DATASETS : ['sample', 'carddemo-seed']


## Copybook decoding

Signed `DISPLAY` numerics use the trailing overpunch (`{`/`A`-`I` positive, `}`/`J`-`R` negative), as produced by GnuCOBOL `-fsign=EBCDIC` and by the Java writer.

In [2]:
from pyspark.sql import functions as F, types as T

POS, NEG = "{ABCDEFGHI", "}JKLMNOPQR"

def zoned_to_decimal(field, scale=2):
    if field is None:
        return None
    last = field[-1]
    if last in POS:
        sign, digit = 1, POS.index(last)
    elif last in NEG:
        sign, digit = -1, NEG.index(last)
    else:
        sign, digit = 1, int(last)
    return Decimal(sign * int(field[:-1] + str(digit))).scaleb(-scale)

zoned_udf = F.udf(zoned_to_decimal, T.DecimalType(13, 2))

TRAN_LAYOUT = [  # (name, offset, length) from CVTRA05Y
    ("tran_id", 0, 16), ("tran_type_cd", 16, 2), ("tran_cat_cd", 18, 4), ("tran_source", 22, 10),
    ("tran_desc", 32, 100), ("tran_amt_raw", 132, 11), ("tran_merchant_id", 143, 9),
    ("tran_merchant_name", 152, 50), ("tran_merchant_city", 202, 50), ("tran_merchant_zip", 252, 10),
    ("tran_card_num", 262, 16), ("tran_orig_ts", 278, 26), ("tran_proc_ts", 304, 26), ("filler", 330, 20)]
ACCT_LAYOUT = [  # from CVACT01Y
    ("acct_id", 0, 11), ("acct_active_status", 11, 1), ("acct_curr_bal_raw", 12, 12),
    ("acct_credit_limit_raw", 24, 12), ("acct_cash_credit_limit_raw", 36, 12),
    ("acct_open_date", 48, 10), ("acct_expiration_date", 58, 10), ("acct_reissue_date", 68, 10),
    ("acct_curr_cyc_credit_raw", 78, 12), ("acct_curr_cyc_debit_raw", 90, 12),
    ("acct_addr_zip", 102, 10), ("acct_group_id", 112, 10), ("filler", 122, 178)]

def read_fixed(path, reclen, layout):
    files = spark.read.format("binaryFile").load(path).select("content")
    def split(row):
        data = bytes(row.content).decode("ascii")
        if len(data) % reclen:
            raise ValueError(f"{path}: size {len(data)} is not a multiple of {reclen}")
        return [(i // reclen + 1, data[i:i + reclen]) for i in range(0, len(data), reclen)]
    schema = T.StructType([T.StructField("seq", T.IntegerType()), T.StructField("record", T.StringType())])
    recs = spark.createDataFrame(files.rdd.flatMap(split), schema)
    return recs.select("seq", "record",
                       *[F.substring("record", off + 1, ln).alias(name) for name, off, ln in layout])

def read_transactions(path):
    df = read_fixed(path, 350, TRAN_LAYOUT)
    return (df.withColumn("tran_amt", zoned_udf("tran_amt_raw"))
              .withColumn("acct_id", F.regexp_extract("tran_desc", r"Int\. for a/c (\d{11})", 1))
              # Hundredths of a second come from the wall clock in CBACT04C (FUNCTION CURRENT-DATE)
              # and cannot be pinned, so timestamps are compared to the second.
              .withColumn("tran_orig_ts_sec", F.substring("tran_orig_ts", 1, 19))
              .withColumn("tran_proc_ts_sec", F.substring("tran_proc_ts", 1, 19)))

def read_accounts(path):
    df = read_fixed(path, 300, ACCT_LAYOUT)
    for c in ["acct_curr_bal", "acct_credit_limit", "acct_cash_credit_limit",
              "acct_curr_cyc_credit", "acct_curr_cyc_debit"]:
        df = df.withColumn(c, zoned_udf(f"{c}_raw"))
    return df

## Comparison

In [3]:
TRAN_COMPARE_COLS = ["seq", "tran_type_cd", "tran_cat_cd", "tran_source", "tran_desc", "tran_amt",
                     "tran_merchant_id", "tran_merchant_name", "tran_merchant_city", "tran_merchant_zip",
                     "tran_card_num", "tran_orig_ts_sec", "tran_proc_ts_sec", "filler"]
ACCT_COMPARE_COLS = ["seq", "record"]

def row_diff(cobol, java, key, cols):
    # Full outer join on the key; one output row per (key, column) that differs.
    j = cobol.select(key, *[F.col(c).alias(f"c_{c}") for c in cols]).join(
        java.select(key, *[F.col(c).alias(f"j_{c}") for c in cols]), key, "full_outer")
    diffs = [F.when(~F.col(f"c_{c}").eqNullSafe(F.col(f"j_{c}")),
                    F.struct(F.lit(c).alias("column"), F.col(f"c_{c}").cast("string").alias("cobol"),
                             F.col(f"j_{c}").cast("string").alias("java"))) for c in cols]
    return (j.select(key, F.explode(F.filter(F.array(*diffs), lambda x: x.isNotNull())).alias("d"))
             .select(key, "d.column", "d.cobol", "d.java"))

results = []
for ds in DATASETS:
    print("\n" + "#" * 78 + f"\n# DATASET: {ds}\n" + "#" * 78)
    c_tran = read_transactions(f"{BASE_PATH}/{ds}/cobol_transact.dat")
    j_tran = read_transactions(f"{BASE_PATH}/{ds}/java/transact.dat")
    c_acct = read_accounts(f"{BASE_PATH}/{ds}/cobol_acctdata.dat")
    j_acct = read_accounts(f"{BASE_PATH}/{ds}/java/acctdata.dat")

    # 1. Row counts
    counts = {"transactions": (c_tran.count(), j_tran.count()), "accounts": (c_acct.count(), j_acct.count())}
    print("\n1. Row counts")
    spark.createDataFrame([(k, v[0], v[1], v[0] == v[1]) for k, v in counts.items()],
                          "file string, cobol long, java long, match boolean").show(truncate=False)

    # 2. Interest per account
    def per_acct(tran, acct, side):
        interest = tran.groupBy("acct_id").agg(F.count("*").alias(f"{side}_txn_count"),
                                               F.sum("tran_amt").alias(f"{side}_interest"))
        return acct.select("acct_id", F.col("acct_curr_bal").alias(f"{side}_curr_bal")).join(
            interest, "acct_id", "left")
    pa = (per_acct(c_tran, c_acct, "cobol").join(per_acct(j_tran, j_acct, "java"), "acct_id", "full_outer")
          .withColumn("match", F.col("cobol_interest").eqNullSafe(F.col("java_interest"))
                      & F.col("cobol_curr_bal").eqNullSafe(F.col("java_curr_bal"))
                      & F.col("cobol_txn_count").eqNullSafe(F.col("java_txn_count")))
          .orderBy("acct_id"))
    print("2. Interest per account (sum of TRAN-AMT) and ACCT-CURR-BAL after the run")
    pa.select("acct_id", "cobol_txn_count", "java_txn_count", "cobol_interest", "java_interest",
              "cobol_curr_bal", "java_curr_bal", "match").show(100, truncate=False)
    acct_mismatch = pa.filter(~F.col("match")).count()

    # 3. Row-level diff
    tran_diff = row_diff(c_tran, j_tran, "tran_id", TRAN_COMPARE_COLS)
    acct_diff = row_diff(c_acct, j_acct, "acct_id", ACCT_COMPARE_COLS)
    n_tran_diff, n_acct_diff = tran_diff.count(), acct_diff.count()
    print(f"3a. Transaction row-level diff (key TRAN-ID, all fields): {n_tran_diff} differing cells")
    tran_diff.show(50, truncate=False)
    print(f"3b. Account row-level diff (key ACCT-ID, full 300-byte record): {n_acct_diff} differing records")
    acct_diff.show(50, truncate=False)
    ts_hundredths_only = (c_tran.join(j_tran, "tran_id")
                          .filter(c_tran.tran_orig_ts != j_tran.tran_orig_ts).count())
    print(f"   (info) transactions whose timestamps differ only below the second: {ts_hundredths_only}")

    passed = (counts["transactions"][0] == counts["transactions"][1]
              and counts["accounts"][0] == counts["accounts"][1]
              and acct_mismatch == 0 and n_tran_diff == 0 and n_acct_diff == 0)
    results.append((ds, counts["transactions"][0], counts["transactions"][1], counts["accounts"][0],
                    counts["accounts"][1], acct_mismatch, n_tran_diff, n_acct_diff,
                    "PASS" if passed else "FAIL"))


##############################################################################
# DATASET: sample
##############################################################################



1. Row counts


+------------+-----+----+-----+
|file        |cobol|java|match|
+------------+-----+----+-----+
|transactions|12   |12  |true |
|accounts    |8    |8   |true |
+------------+-----+----+-----+

2. Interest per account (sum of TRAN-AMT) and ACCT-CURR-BAL after the run


+-----------+---------------+--------------+--------------+-------------+--------------+-------------+-----+
|acct_id    |cobol_txn_count|java_txn_count|cobol_interest|java_interest|cobol_curr_bal|java_curr_bal|match|
+-----------+---------------+--------------+--------------+-------------+--------------+-------------+-----+
|00000000101|3              |3             |50.28         |50.28        |1550.28       |1550.28      |true |
|00000000102|3              |3             |-3.13         |-3.13        |-253.88       |-253.88      |true |
|00000000103|NULL           |NULL          |NULL          |NULL         |5000.00       |5000.00      |true |
|00000000104|2              |2             |1.66          |1.66         |801.66        |801.66       |true |
|00000000105|2              |2             |22881584.33   |22881584.33  |1022881584.32 |1022881584.32|true |
|00000000106|1              |1             |7.77          |7.77         |7.77          |7.77         |true |
|00000000107|NULL  

3a. Transaction row-level diff (key TRAN-ID, all fields): 0 differing cells


+-------+------+-----+----+
|tran_id|column|cobol|java|
+-------+------+-----+----+
+-------+------+-----+----+

3b. Account row-level diff (key ACCT-ID, full 300-byte record): 0 differing records
+-------+------+-----+----+
|acct_id|column|cobol|java|
+-------+------+-----+----+
+-------+------+-----+----+



... (2 outputs left out)

2. Interest per account (sum of TRAN-AMT) and ACCT-CURR-BAL after the run


+-----------+---------------+--------------+--------------+-------------+--------------+-------------+-----+
|acct_id    |cobol_txn_count|java_txn_count|cobol_interest|java_interest|cobol_curr_bal|java_curr_bal|match|
+-----------+---------------+--------------+--------------+-------------+--------------+-------------+-----+
|00000000001|1              |1             |0.00          |0.00         |194.00        |194.00       |true |
|00000000002|1              |1             |0.00          |0.00         |158.00        |158.00       |true |
|00000000003|1              |1             |0.00          |0.00         |147.00        |147.00       |true |
|00000000004|1              |1             |0.00          |0.00         |40.00         |40.00        |true |
|00000000005|1              |1             |0.00          |0.00         |345.00        |345.00       |true |
|00000000006|1              |1             |0.00          |0.00         |218.00        |218.00       |true |
|00000000007|1     

3a. Transaction row-level diff (key TRAN-ID, all fields): 0 differing cells
+-------+------+-----+----+
|tran_id|column|cobol|java|
+-------+------+-----+----+
+-------+------+-----+----+

3b. Account row-level diff (key ACCT-ID, full 300-byte record): 0 differing records


+-------+------+-----+----+
|acct_id|column|cobol|java|
+-------+------+-----+----+
+-------+------+-----+----+

   (info) transactions whose timestamps differ only below the second: 50


## Verdict

In [4]:
summary = spark.createDataFrame(results, "dataset string, cobol_txns long, java_txns long, cobol_accts long, "
                                "java_accts long, acct_mismatches long, txn_cell_diffs long, "
                                "acct_record_diffs long, result string")
summary.show(truncate=False)
OVERALL = "PASS" if all(r[-1] == "PASS" for r in results) else "FAIL"
print("=" * 78)
print(f"OVERALL RESULT: {OVERALL}    [{RUN_MODE}]")
print("=" * 78)
assert OVERALL == "PASS", "COBOL and Java outputs differ"

+-------------+----------+---------+-----------+----------+---------------+--------------+-----------------+------+
|dataset      |cobol_txns|java_txns|cobol_accts|java_accts|acct_mismatches|txn_cell_diffs|acct_record_diffs|result|
+-------------+----------+---------+-----------+----------+---------------+--------------+-----------------+------+
|sample       |12        |12       |8          |8         |0              |0             |0                |PASS  |
|carddemo-seed|50        |50       |50         |50        |0              |0             |0                |PASS  |
+-------------+----------+---------+-----------+----------+---------------+--------------+-----------------+------+

OVERALL RESULT: PASS    [LOCAL RUN - PySpark 3.5.1 on this machine, NOT Databricks]
